# Exact flip-odd orbit fold: fail-closed Colab verification

This notebook verifies one **WIP/preregistered** exact finite-sum theorem at an immutable public SHA. It is not a certificate by itself. It fails before compilation if the repository SHA, Lean toolchain, Mathlib pin, gate hash, gate counters, or normal/optimized gate output differs from the registered values. A final `PASS` is printed only after the gate, module build, core build, permanent oracle, and consistency judge all succeed.

The target remains: for `beta >= 0`, `gamma >= 0`, every extent, `specRatio <= tanh(beta) * exp(2*gamma)`. This notebook validates only the lossless orbit fold of an arbitrary flip-odd action. It does not prove a norm estimate, spectral classification, either analytic sector bound, or progress on that target.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, re, shutil, subprocess, sys, tempfile

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
EXPECTED_SHA = 'fca2f37705bc06cf659829bde64ea0fd5c810638'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_LEAN_COMMIT = '00659f8e6071d7e46131ed643bf8003b99b044e9'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
EXPECTED_GATE_SHA256 = '43d218195d497f65d8e6012081a36215a363574fbf3046dccc3eec01eb18ab89'
EXPECTED_CORE_JOBS = 8466
RUN_ROOT = Path(tempfile.mkdtemp(prefix='spatial-odd-orbit-fold-'))
REPO = RUN_ROOT / 'repo'
ARTIFACTS = RUN_ROOT / 'artifacts'
ARTIFACTS.mkdir()
TRANSCRIPT = ARTIFACTS / 'transcript.txt'

def log(text):
    text = str(text)
    print(text)
    with TRANSCRIPT.open('a', encoding='utf-8', newline='\n') as f:
        f.write(text + '\n')

def run(cmd, cwd=None, env=None, allow_failure=False):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    try:
        p = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    except FileNotFoundError as exc:
        log(f'[missing executable: {exc.filename}]')
        log('[exit 127]')
        if allow_failure:
            return subprocess.CompletedProcess(cmd, 127, stdout=str(exc))
        raise
    log(p.stdout.rstrip())
    log(f'[exit {p.returncode}]')
    if p.returncode and not allow_failure:
        raise RuntimeError(f'command failed ({p.returncode}): {shown}')
    return p

log('SPATIAL ODD ORBIT-FOLD LEAN COLAB RUN')
log(f'utc_start={datetime.datetime.now(datetime.timezone.utc).isoformat()}')
log(f'platform={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').splitlines()[4] if Path('/proc/cpuinfo').exists() else 'cpuinfo=unavailable')
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0] if Path('/proc/meminfo').exists() else 'meminfo=unavailable')
log('gpu=none requested or allocated')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
actual_sha = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip()
if actual_sha != EXPECTED_SHA:
    raise RuntimeError(f'SHA mismatch: {actual_sha} != {EXPECTED_SHA}')
toolchain = (REPO / 'lean-toolchain').read_text(encoding='utf-8').strip()
if toolchain != EXPECTED_TOOLCHAIN:
    raise RuntimeError(f'toolchain mismatch: {toolchain} != {EXPECTED_TOOLCHAIN}')
manifest = json.loads((REPO / 'lake-manifest.json').read_text(encoding='utf-8'))
mathlib_entries = [p for p in manifest['packages'] if p.get('name') == 'mathlib']
if len(mathlib_entries) != 1:
    raise RuntimeError(f'expected exactly one mathlib manifest entry, got {len(mathlib_entries)}')
manifest_rev = mathlib_entries[0].get('rev')
if manifest_rev != EXPECTED_MATHLIB:
    raise RuntimeError(f'mathlib mismatch: {manifest_rev} != {EXPECTED_MATHLIB}')
log(f'repo_sha={actual_sha}')
log(f'lean_toolchain={toolchain}')
gate = REPO / 'scripts' / 'judge_spatial_odd_orbit_fold.py'
gate_hash = hashlib.sha256(gate.read_bytes()).hexdigest()
if gate_hash != EXPECTED_GATE_SHA256:
    raise RuntimeError(f'gate SHA-256 mismatch: {gate_hash}')
expected_gate = {
  'status': 'PASS', 'classification': 'exact flip-odd orbit-folding gate only',
  'ring_sizes': [1, 2, 3, 4, 5, 6, 7], 'action_rows_checked': 254,
  'paired_summands_checked': 10922, 'sign_mutations_rejected': 10922,
  'omission_mutations_rejected': 10922, 'head_only_flip_mutations_rejected': 10920,
}
gate_outputs = []
for flags in ([], ['-O']):
    gate_run = run([sys.executable, *flags, str(gate)], cwd=REPO)
    lines = gate_run.stdout.splitlines()
    if len(lines) != 1:
        raise RuntimeError(f'gate emitted stale or extra output under flags {flags}')
    payload = json.loads(lines[0])
    if payload != expected_gate:
        raise RuntimeError(f'gate output mismatch under flags {flags}: {payload}')
    gate_outputs.append(lines[0])
if gate_outputs[0] != gate_outputs[1]:
    raise RuntimeError('normal and optimized gate outputs diverged')
log(f'mathlib_pin={manifest_rev}')
log(f'gate_sha256={gate_hash}')
log('PRECHECK_AND_GATE PASS')

In [ ]:
elan_home = RUN_ROOT / 'elan'
env = os.environ.copy()
env['ELAN_HOME'] = str(elan_home)
env['PATH'] = str(elan_home / 'bin') + os.pathsep + env['PATH']
installer = RUN_ROOT / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh', '-o', str(installer)])
installer_sha = hashlib.sha256(installer.read_bytes()).hexdigest()
log(f'elan_installer_sha256={installer_sha}')
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'], env=env)
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
lean_version = run(['lean', '--version'], cwd=REPO, env=env).stdout
if 'version 4.29.0-rc6' not in lean_version or EXPECTED_LEAN_COMMIT not in lean_version:
    raise RuntimeError(f'Lean binary mismatch: {lean_version}')
run(['lake', '--version'], cwd=REPO, env=env)
# Official Mathlib cache is permitted only inside this isolated ephemeral runtime.
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
log('TOOLCHAIN_AND_CACHE PASS')

In [ ]:
new_declarations = ['act_flipOdd_eq_sum_zero_head']
oracle = REPO / 'oracle_check.lean'
module = run(['lake', 'build', 'YangMills.OS.SpatialRing'], cwd=REPO, env=env)
module_match = re.search(r'Build completed successfully \((\d+) jobs\)', module.stdout)
if not module_match:
    raise RuntimeError('module build succeeded without a measured job count')
core = run(['lake', 'build', 'YangMillsCore'], cwd=REPO, env=env)
match = re.search(r'Build completed successfully \((\d+) jobs\)', core.stdout)
if not match:
    raise RuntimeError('full core build succeeded without a measured job count')
core_jobs = int(match.group(1))
if core_jobs != EXPECTED_CORE_JOBS:
    raise RuntimeError(f'core job-count mismatch: {core_jobs}')
oracle_run = run(['lake', 'env', 'lean', str(oracle)], cwd=REPO, env=env)
for name in new_declarations:
    marker = f"'YangMills.OS.{name}' depends on axioms:"
    if marker not in oracle_run.stdout:
        raise RuntimeError(f'permanent oracle omitted output for {name}')
allowed_axioms = {'propext', 'Classical.choice', 'Quot.sound'}
if 'sorryAx' in oracle_run.stdout:
    raise RuntimeError('oracle contains sorryAx')
for line in oracle_run.stdout.splitlines():
    if 'depends on axioms:' not in line:
        continue
    payload = line.split('depends on axioms:', 1)[1].strip().strip('[]')
    used = {x.strip() for x in payload.split(',') if x.strip()}
    if not used <= allowed_axioms:
        raise RuntimeError(f'nonstandard axioms: {used - allowed_axioms}')
run(['python3', 'scripts/check_consistency.py'], cwd=REPO, env=env)
metadata = {
  'repo_sha': actual_sha, 'toolchain': toolchain, 'lean_commit': EXPECTED_LEAN_COMMIT,
  'mathlib_pin': manifest_rev, 'gate_sha256': gate_hash,
  'module_jobs': int(module_match.group(1)), 'core_jobs': core_jobs,
  'utc_end': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'cpu_count': os.cpu_count(),
  'memory': Path('/proc/meminfo').read_text(errors='replace').splitlines()[0],
  'elan_installer_sha256': installer_sha,
}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
hashes = {}
for path in sorted(ARTIFACTS.iterdir()):
    if path.name != 'SHA256SUMS':
        hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
(ARTIFACTS / 'SHA256SUMS').write_text(''.join(f'{h}  {name}\n' for name, h in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_odd_orbit_fold_lean_artifacts', 'zip', ARTIFACTS)
log(f'module_jobs_measured={module_match.group(1)}')
log(f'core_jobs_measured={core_jobs}')
log(f'artifact_zip={archive}')
log(f'artifact_zip_sha256={hashlib.sha256(Path(archive).read_bytes()).hexdigest()}')
log('SPATIAL ODD ORBIT-FOLD LEAN PASS')
from google.colab import files
files.download(archive)